# Murphy 1 · Chapter 8 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M8). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 8.1 · When does gradient descent blow up?  
*Section §8.2 · code*

Minimise f(w) = ½(w₁² + 10w₂²) from w = [5, 5] with gradient descent for 100 steps, for step sizes 0.05, 0.15, 0.19, 0.21. Print the final loss for each. Predict the critical step size first.

<details><summary><b>Hint</b></summary>

The curvatures are 1 and 10. Along a direction with curvature c, each step multiplies the error by (1 − ηc).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
c = np.array([1., 10.])
for eta in [0.05, 0.15, 0.19, 0.21]:
    w = np.array([5., 5.])
    for _ in range(100):
        w = ...  # TODO: gradient of 0.5*sum(c*w^2) is c*w
    print(f"eta {eta}: final loss {0.5 * np.sum(c * w**2):.3e}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
c = np.array([1., 10.])
for eta in [0.05, 0.15, 0.19, 0.21]:
    w = np.array([5., 5.])
    for _ in range(100):
        w = w - eta * c * w  # gradient of 0.5*sum(c*w^2) is c*w
    print(f"eta {eta}: final loss {0.5 * np.sum(c * w**2):.3e}")

<details><summary><b>Answer and explanation</b></summary>

Along the steep direction the error is multiplied by (1 − 10η) each step. That shrinks only if |1 − 10η| < 1, i.e. η < 0.2. At 0.19 it converges slowly with zig-zags; at 0.21 it explodes. The steep direction sets the speed limit, while the flat direction (curvature 1) is what makes convergence slow (Murphy §8.2.2–8.2.3).

Expected output:
```
eta 0.05: final loss 4.382e-04
eta 0.15: final loss 9.565e-14
eta 0.19: final loss 8.819e-08
eta 0.21: final loss 2.374e+10
```

</details>

## 8.2 · Momentum and Adam on the banana  
*Section §8.2 · code*

Implement plain GD, momentum (β = 0.9) and Adam on the Rosenbrock function f = (1 − x)² + 100(y − x²)² starting at (−1.5, 2). Run 5,000 steps each and report the final point and loss.

<details><summary><b>Hint</b></summary>

Gradient: ∂f/∂x = −2(1 − x) − 400x(y − x²), ∂f/∂y = 200(y − x²).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
f = lambda p: (1 - p[0])**2 + 100 * (p[1] - p[0]**2)**2
g = lambda p: np.array([-2*(1 - p[0]) - 400*p[0]*(p[1] - p[0]**2), 200*(p[1] - p[0]**2)])
start = np.array([-1.5, 2.0])
p = start.copy()
for _ in range(5000): p -= 1e-4 * g(p)
print("GD       ", p.round(4), f"{f(p):.2e}")
p, v = start.copy(), np.zeros(2)
for _ in range(5000):
    v = ...  # TODO: keep 90% of the previous velocity
    p -= 1e-4 * v
print("momentum ", p.round(4), f"{f(p):.2e}")
p, m, s = start.copy(), np.zeros(2), np.zeros(2)
for t in range(1, 5001):
    gr = g(p); m = 0.9*m + 0.1*gr; s = 0.999*s + 0.001*gr**2
    p -= ...  # TODO: averaged slope / its typical size
print("Adam     ", p.round(4), f"{f(p):.2e}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
f = lambda p: (1 - p[0])**2 + 100 * (p[1] - p[0]**2)**2
g = lambda p: np.array([-2*(1 - p[0]) - 400*p[0]*(p[1] - p[0]**2), 200*(p[1] - p[0]**2)])
start = np.array([-1.5, 2.0])
p = start.copy()
for _ in range(5000): p -= 1e-4 * g(p)
print("GD       ", p.round(4), f"{f(p):.2e}")
p, v = start.copy(), np.zeros(2)
for _ in range(5000):
    v = 0.9 * v + g(p)  # keep 90% of the previous velocity
    p -= 1e-4 * v
print("momentum ", p.round(4), f"{f(p):.2e}")
p, m, s = start.copy(), np.zeros(2), np.zeros(2)
for t in range(1, 5001):
    gr = g(p); m = 0.9*m + 0.1*gr; s = 0.999*s + 0.001*gr**2
    p -= 0.01 * (m / (1 - 0.9**t)) / (np.sqrt(s / (1 - 0.999**t)) + 1e-8)  # averaged slope / its typical size
print("Adam     ", p.round(4), f"{f(p):.2e}")

<details><summary><b>Answer and explanation</b></summary>

The minimum is at (1, 1). GD with a safe small step crawls along the curved valley floor and is still far away after 5,000 steps. Momentum builds speed along the valley and gets much closer. Adam's per-direction scaling handles the steep walls and the flat floor at the same time.

Expected output:
```
GD        [-1.1162  1.2539] 4.48e+00
momentum  [0.9202 0.8464] 6.38e-03
Adam      [1. 1.] 1.68e-17
```

</details>

## 8.3 · SGD with different batch sizes  
*Section §8.4 · code*

Fit linear regression (10,000 points, 5 features) with minibatch SGD for one fixed budget of 20,000 gradient evaluations, using batch sizes 1, 10, 100. Compare the final distance to the true weights.

<details><summary><b>Hint</b></summary>

Steps = budget / batch size. Use the same learning rate 0.01.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(12)
X = rng.normal(size=(10000, 5)); w_true = rng.normal(size=5)
y = X @ w_true + 0.5 * rng.normal(size=10000)
for B in [1, 10, 100]:
    w = np.zeros(5)
    for _ in range(20000 // B):
        i = rng.integers(0, 10000, B)
        grad = ...  # TODO: minibatch gradient of the squared error
        w -= 0.01 * grad
    print(f"batch {B:3d}: steps {20000//B:5d}  |w - w_true| = {np.linalg.norm(w - w_true):.4f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(12)
X = rng.normal(size=(10000, 5)); w_true = rng.normal(size=5)
y = X @ w_true + 0.5 * rng.normal(size=10000)
for B in [1, 10, 100]:
    w = np.zeros(5)
    for _ in range(20000 // B):
        i = rng.integers(0, 10000, B)
        grad = 2 * X[i].T @ (X[i] @ w - y[i]) / B  # minibatch gradient of the squared error
        w -= 0.01 * grad
    print(f"batch {B:3d}: steps {20000//B:5d}  |w - w_true| = {np.linalg.norm(w - w_true):.4f}")

<details><summary><b>Answer and explanation</b></summary>

Batch 1 takes 20,000 very noisy steps: it gets near the answer fast but keeps rattling around it (error ≈ 0.11). Batch 100 takes clean steps but only 200 of them, so it hasn't finished (≈ 0.05). Batch 10 is the sweet spot here (≈ 0.03). Gradient noise shrinks like 1/√B (the CLT lab), so bigger batches buy smoothness with fewer steps. That trade-off is why typical batch sizes are in the tens to hundreds.

Expected output:
```
batch   1: steps 20000  |w - w_true| = 0.1091
batch  10: steps  2000  |w - w_true| = 0.0279
batch 100: steps   200  |w - w_true| = 0.0472
```

</details>

## 8.4 · EM for a two-bump mixture  
*Section §8.7 · code*

Generate 600 points: 40% from N(−2, 0.7²) and 60% from N(3, 1.2²). Fit a 2-component Gaussian mixture with EM, starting from means [−1, 1], σ = [1, 1], weights [0.5, 0.5]. Print the parameters after 50 iterations.

<details><summary><b>Hint</b></summary>

E step: r_ik ∝ π_k N(x_i | μ_k, σ_k). M step: π_k = mean_i r_ik, μ_k = Σ r_ik x_i / Σ r_ik, σ_k² = Σ r_ik (x_i − μ_k)² / Σ r_ik.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(13)
x = np.concatenate([rng.normal(-2, 0.7, 240), rng.normal(3, 1.2, 360)])
pi, mu, sd = np.array([.5, .5]), np.array([-1., 1.]), np.array([1., 1.])
N = lambda x, m, s: np.exp(-(x - m)**2 / (2 * s**2)) / (s * np.sqrt(2 * np.pi))
for _ in range(50):
    r = pi * N(x[:, None], mu, sd)          # E step: unnormalised responsibilities
    r /= ...  # TODO: normalise each row to sum to 1
    nk = r.sum(0)
    pi = ...  # TODO: M step: new weights
    mu = ...  # TODO: new means
    sd = ...  # TODO: new spreads
print("weights", pi.round(3), " means", mu.round(3), " sds", sd.round(3))

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(13)
x = np.concatenate([rng.normal(-2, 0.7, 240), rng.normal(3, 1.2, 360)])
pi, mu, sd = np.array([.5, .5]), np.array([-1., 1.]), np.array([1., 1.])
N = lambda x, m, s: np.exp(-(x - m)**2 / (2 * s**2)) / (s * np.sqrt(2 * np.pi))
for _ in range(50):
    r = pi * N(x[:, None], mu, sd)          # E step: unnormalised responsibilities
    r /= r.sum(1, keepdims=True)  # normalise each row to sum to 1
    nk = r.sum(0)
    pi = nk / len(x)  # M step: new weights
    mu = (r * x[:, None]).sum(0) / nk  # new means
    sd = np.sqrt((r * (x[:, None] - mu)**2).sum(0) / nk)  # new spreads
print("weights", pi.round(3), " means", mu.round(3), " sds", sd.round(3))

<details><summary><b>Answer and explanation</b></summary>

EM recovers weights ≈ [0.4, 0.6], means ≈ [−2, 3], σ ≈ [0.7, 1.2]. The E step makes soft guesses of which bump each point came from; the M step refits each bump to the points weighted by those guesses. The log-likelihood never decreases.

Expected output:
```
weights [0.405 0.595]  means [-1.929  3.049]  sds [0.777 1.202]
```

</details>

## 8.5 · Newton's method by hand  
*Section §8.3 · derive*

Minimise f(x) = x⁴ − 3x² + 2 with Newton's method starting at x₀ = 2. Do two steps by hand.

<details><summary><b>Hint</b></summary>

x ← x − f′(x)/f″(x), with f′ = 4x³ − 6x and f″ = 12x² − 6.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

x₀ = 2: f′ = 32 − 12 = 20, f″ = 48 − 6 = 42, x₁ = 2 − 20/42 ≈ 1.524. x₁: f′ = 4(3.540) − 9.143 ≈ 5.016, f″ = 12(2.322) − 6 ≈ 21.87, x₂ ≈ 1.524 − 0.229 ≈ 1.294. The true minimum is √1.5 ≈ 1.225, and Newton converges quadratically: the number of correct digits roughly doubles each step once close.

</details>